# Earthquake Damage Prediction — Complete Notebook
This notebook loads the uploaded dataset, performs EDA, preprocessing, model comparison (RandomForest, XGBoost), evaluation, and saves the final models.

If any required package is missing, the notebook will install it. Run cells sequentially.

Files expected (automatically searched in `/mnt/data`):
- `PRCP-1015-EquakeDamagePred.docx` (project brief)
- `PRCP-1015-EquakeDamagePred (2).zip` or similar zip containing dataset

If your dataset is already extracted, set `data_dir` accordingly in the Data Loading cell.

In [ ]:
### Install required packages (runs only if packages missing)
import sys
import subprocess
def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages)

needed = []
try:
    import xgboost
except Exception:
    needed.append('xgboost')
try:
    import lightgbm
except Exception:
    needed.append('lightgbm')
try:
    import pandas
except Exception:
    needed.append('pandas')
try:
    import sklearn
except Exception:
    needed.append('scikit-learn')
if needed:
    print('Installing:', needed)
    pip_install(needed)
else:
    print('All packages present.')


All packages present.


In [ ]:
### Data loading
import os, zipfile, glob
base = ''
# Try to find a zip file uploaded by user
zip_paths = glob.glob(os.path.join(base, '*.zip')) + glob.glob(os.path.join(base, '*EquakeDamagePred*.zip'))
print('Found zip files:', zip_paths)
data_dir = os.path.join(base, 'equake_data')
os.makedirs(data_dir, exist_ok=True)
if zip_paths:
    z = zip_paths[0]
    print('Extracting', z)
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
else:
    print('No zip found in /mnt/data — if dataset is already extracted, set data_dir to the folder containing csv files.')

# Look for CSV files
print(data_dir)
csvs = glob.glob(os.path.join(data_dir,'Data', '*.csv')) + glob.glob(os.path.join(base, '*.csv'))
print(csvs)
print('CSV files found:', csvs)
df = pd.DataFrame()
if not csvs:
    print('\nIf you see no CSVs, unzip the dataset into /mnt/data or upload the dataset CSVs to the environment.')
else:
    # Attempt to load train and test if present
    import pandas as pd
    train_path = None
    for p in csvs:
        if 'train' in os.path.basename(p).lower():
            train_path = p

    if train_path is None:
        train_path = csvs[0]
    print('Using train file:', train_path)
    temp1= pd.read_csv(csvs[0])
    temp2 = pd.read_csv(csvs[1])
    df = pd.merge(temp1,temp2,on="building_id",how="inner")
    print('Loaded dataframe shape:', df.shape)


Found zip files: ['PRCP-1015-EquakeDamagePred.zip', 'PRCP-1015-EquakeDamagePred.zip']
Extracting PRCP-1015-EquakeDamagePred.zip
equake_data
['equake_data/Data/train_values.csv', 'equake_data/Data/train_labels.csv']
CSV files found: ['equake_data/Data/train_values.csv', 'equake_data/Data/train_labels.csv']
Using train file: equake_data/Data/train_labels.csv
Loaded dataframe shape: (260601, 40)


In [ ]:
### Quick EDA
import pandas as pd
try:
    df
except NameError:
    raise FileNotFoundError('Dataframe `df` not found. Run the Data loading cell and ensure a CSV exists in /mnt/data or equake_data.')
display(df.head())
print('\nInfo:')
print(df.info())
print('\nMissing values per column:')
print(df.isnull().sum().sort_values(ascending=False).head(30))


,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
0,802906,6,487,12198,2,30,6,5,t,r,...,0,0,0,0,0,0,0,0,0,3
1,28830,8,900,2812,2,10,8,7,o,r,...,0,0,0,0,0,0,0,0,0,2
2,94947,21,363,8973,2,10,5,5,t,r,...,0,0,0,0,0,0,0,0,0,3
3,590882,22,418,10694,2,10,6,5,t,r,...,0,0,0,0,0,0,0,0,0,2
4,201944,11,131,1488,3,30,8,9,t,r,...,0,0,0,0,0,0,0,0,0,3



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260601 entries, 0 to 260600
Data columns (total 40 columns):
 #   Column                                  Non-Null Count   Dtype 
---  ------                                  --------------   ----- 
 0   building_id                             260601 non-null  int64 
 1   geo_level_1_id                          260601 non-null  int64 
 2   geo_level_2_id                          260601 non-null  int64 
 3   geo_level_3_id                          260601 non-null  int64 
 4   count_floors_pre_eq                     260601 non-null  int64 
 5   age                                     260601 non-null  int64 
 6   area_percentage                         260601 non-null  int64 
 7   height_percentage                       260601 non-null  int64 
 8   land_surface_condition                  260601 non-null  object
 9   foundation_type                         260601 non-null  object
 10  roof_type                               260601 no

In [ ]:
df.head()

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
0,802906,6,487,12198,2,30,6,5,t,r,...,0,0,0,0,0,0,0,0,0,3
1,28830,8,900,2812,2,10,8,7,o,r,...,0,0,0,0,0,0,0,0,0,2
2,94947,21,363,8973,2,10,5,5,t,r,...,0,0,0,0,0,0,0,0,0,3
3,590882,22,418,10694,2,10,6,5,t,r,...,0,0,0,0,0,0,0,0,0,2
4,201944,11,131,1488,3,30,8,9,t,r,...,0,0,0,0,0,0,0,0,0,3


In [ ]:
### Preprocessing
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Basic cleanup: drop columns with single unique value or id column
df = df.copy()
if 'building_id' in df.columns:
    df.drop('building_id', axis=1, inplace=True)

# Target
if 'damage_grade' not in df.columns:
    raise KeyError('Target column `damage_grade` not found in data. Please ensure the CSV contains this column.')
y = df['damage_grade']
X = df.drop('damage_grade', axis=1)

# Encode categorical columns that are object dtype
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns detected:', cat_cols)
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = X[c].fillna('NA')
    X[c] = le.fit_transform(X[c].astype(str))
    le_dict[c] = le

# Fill numeric missing with median
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols:
    X[c] = X[c].fillna(X[c].median())

# Train test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


Categorical columns detected: ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
Train shape: (208480, 38) Test shape: (52121, 38)


In [ ]:
X_train.head()

,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,roof_type,...,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other
185461,21,935,5965,3,20,6,6,2,2,0,...,0,0,0,0,0,0,0,0,0,0
190537,3,1229,1099,2,20,6,5,2,2,0,...,0,0,0,0,0,0,0,0,0,0
232781,21,1219,2705,3,45,6,5,2,2,1,...,0,0,0,0,0,0,0,0,0,0
213004,20,158,2452,2,10,7,5,2,4,1,...,0,0,0,0,0,0,0,0,0,0
177786,9,670,5977,2,0,9,6,2,1,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
### Modeling — RandomForest & XGBoost (basic)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# RandomForest baseline
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('RandomForest Classification Report:')
print(classification_report(y_test, y_pred_rf))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_rf))

# XGBoost baseline
import xgboost as xgb
xgb_clf = xgb.XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print('\nXGBoost Classification Report:')
print(classification_report(y_test, y_pred_xgb))




RandomForest Classification Report:
              precision    recall  f1-score   support

           1       0.65      0.49      0.56      5025
           2       0.73      0.83      0.78     29652
           3       0.73      0.62      0.67     17444

    accuracy                           0.72     52121
   macro avg       0.70      0.64      0.67     52121
weighted avg       0.72      0.72      0.72     52121

Confusion matrix:
[[ 2444  2487    94]
 [ 1193 24536  3923]
 [  125  6574 10745]]


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [1 2 3]

In [ ]:
### Optional: Quick hyperparameter tuning for XGBoost (runs only if xgboost present)
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
try:
    import xgboost as xgb
    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.6, 0.8, 1.0]
    }
    xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
    rs = RandomizedSearchCV(xgb_model, param_dist, n_iter=6, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42, verbose=1)
    rs.fit(X_train, y_train)
    print('Best params:', rs.best_params_)
    best_xgb = rs.best_estimator_
    print('\nEvaluation of best XGBoost:')
    from sklearn.metrics import classification_report
    y_pred_best = best_xgb.predict(X_test)
    print(classification_report(y_test, y_pred_best))
except Exception as e:
    print('Skipping hyperparameter tuning — xgboost not available or failed:', e)


In [ ]:
### Save trained models and encoders
import joblib
os.makedirs('/mnt/data/models', exist_ok=True)
joblib.dump(rf, '/mnt/data/models/random_forest.pkl')
print('Saved RandomForest to /mnt/data/models/random_forest.pkl')
try:
    if 'best_xgb' in globals():
        joblib.dump(best_xgb, '/mnt/data/models/xgb_best.pkl')
        print('Saved best XGBoost to /mnt/data/models/xgb_best.pkl')
    elif 'xgb_clf' in globals():
        joblib.dump(xgb_clf, '/mnt/data/models/xgb_baseline.pkl')
        print('Saved XGBoost baseline to /mnt/data/models/xgb_baseline.pkl')
except Exception as e:
    print('Could not save xgboost model:', e)

# Save label encoders if any
joblib.dump(le_dict, '/mnt/data/models/label_encoders.pkl')
print('Saved label encoders to /mnt/data/models/label_encoders.pkl')


## Suggestions for Seismologists & City Planners
- Improve building codes and enforce retrofitting of vulnerable masonry buildings.
- Prioritize inspections in regions with older buildings (higher `age`) and non-engineered RC.
- Use targeted awareness and zoning regulations in high risk `geo_level` regions.

## Challenges faced (sample)
- Imbalanced classes and ordinal nature of `damage_grade` — consider ordinal regression or specialized loss.
- Many categorical features with obfuscated labels requiring careful encoding.
- Possible geographic clustering — consider geo-aware cross-validation.

----
Notebook created automatically. If you'd like, I can now:
1. Add more visualizations (correlation heatmap, per-feature distributions).
2. Replace LabelEncoder -> OneHot for certain cols.
3. Add cross-validation and model logging.
